# vedit · распознавание речи на GPU Colab

Runtime → Change runtime type → **T4 GPU**. Затем Runtime → Run all.

Нужны два файла из проекта: `tools/asr/vedit_asr.py` и `workspace/<job>/asr_upload.ogg` (его делает `vedit asr-export <job>`).
В конце скачается `asr_bundle.zip` → `vedit asr-import <job> asr_bundle.zip` → `vedit understand <job> --asr import`.

In [ ]:
!nvidia-smi -L
!pip -q install faster-whisper speechbrain scipy 2>&1 | tail -1
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# загрузите vedit_asr.py и asr_upload.ogg (можно оба сразу)
from google.colab import files
up = files.upload()
import os
assert os.path.exists('vedit_asr.py'), 'нужен vedit_asr.py'
ogg = [n for n in up if n.endswith('.ogg') or n.endswith('.opus') or n.endswith('.wav') or n.endswith('.mp3') or n.endswith('.m4a')]
assert ogg, 'нужен asr_upload.ogg (vedit asr-export <job>)'
AUDIO = ogg[0]; print('audio:', AUDIO)

In [ ]:
# 16 кГц моно WAV для сайдкара
!ffmpeg -hide_banner -loglevel error -y -i "$AUDIO" -ac 1 -ar 16000 -c:a pcm_s16le audio.wav && ls -la audio.wav

In [ ]:
# speech → faster.json, phrases.json, speakers_raw.json (всё на CUDA)
!python vedit_asr.py pipeline --audio audio.wav --out-dir bundle --device cuda --compute float16 --language ru
# для смешанных или не-русских видео уберите --language ru (определится по первым 30 с)

In [ ]:
import json
d = json.load(open('bundle/faster.json', encoding='utf-8'))
print(d['language'], d['device'], d['compute_type'], f"{d['wall_seconds']:.0f}s wall,", len(d['segments']), 'segments')
for s in d['segments'][:5]: print(f"{s['start']:7.1f}-{s['end']:7.1f}  {s['text']}")
try:
    sp = json.load(open('bundle/speakers_raw.json', encoding='utf-8'))
    print('speakers:', [(s['id'], s['seconds']) for s in sp['speakers']])
except FileNotFoundError:
    print('speakers: skipped')

In [ ]:
!cd bundle && zip -q -r ../asr_bundle.zip faster.json phrases.json speakers_raw.json 2>/dev/null || (cd bundle && zip -q -r ../asr_bundle.zip faster.json phrases.json)
from google.colab import files
files.download('asr_bundle.zip')